# Prueba de conexión Jupyter ↔ HDFS
Comprueba que este Jupyter puede **leer** los CSV de `/datalake/raw` y **escribir** Parquet en HDFS (WebHDFS). Ejecuta las celdas en orden.

In [ ]:
import os
import pandas as pd

# Dirección de HDFS por WebHDFS (viene de las variables de entorno del contenedor)
HOST = os.environ.get('HDFS_WEBHDFS_URL', 'http://namenode:9870').replace('http://', '')
OPT = {'user': os.environ.get('HDFS_USER', 'root')}   # usuario con el que se accede a HDFS
RAW = f'webhdfs://{HOST}/datalake/raw'
CLEAN = f'webhdfs://{HOST}/datalake/clean'
print(RAW)

## 1. Leer un CSV desde HDFS

In [ ]:
# Leemos solo 5 filas del CSV en bruto. dtype=str porque los datos están sucios.
df = pd.read_csv(f'{RAW}/historico/afluencia_diaria_historico.csv', storage_options=OPT, nrows=5, dtype=str)
df

## 2. Escribir un Parquet en HDFS

In [ ]:
# Escribimos esas 5 filas en una carpeta de prueba dentro de /datalake/clean
df.to_parquet(f'{CLEAN}/_prueba/test.parquet', storage_options=OPT, index=False)
print('Parquet escrito')

## 3. Leerlo de vuelta y comprobar

In [ ]:
df2 = pd.read_parquet(f'{CLEAN}/_prueba/test.parquet', storage_options=OPT)
print(df2.shape, '- iguales:', df.equals(df2))
df2.dtypes

## 4. Borrar el archivo de prueba

In [ ]:
import fsspec
fs = fsspec.filesystem('webhdfs', host=HOST.split(':')[0], port=int(HOST.split(':')[1]), **OPT)
fs.rm(f'/datalake/clean/_prueba', recursive=True)
print('Prueba borrada:', not fs.exists('/datalake/clean/_prueba'))